# 18 · Hybrid Embedding Benchmark Walkthrough (v0.3.0)

This notebook walks through the v0.3.0 **3-row retrieval ablation** on the ICD-10-CM → SNOMED concept-mapping benchmark.

**Important:** the *published numbers* are computed against a full Athena vocabulary export (~700 MB once extracted) and the bundled `_demo_data/vocabulary` is intentionally tiny (ICD-10-CM only, no SNOMED). So we won't *run* the benchmark here — we'll inspect the published results, walk through the CLI, and explain the honest finding that BM25 beats hybrid on this task.

For an end-to-end run, see [docs/benchmarks/athena-icd-snomed.md](../benchmarks/athena-icd-snomed.md).

In [ ]:
import json
from pathlib import Path

import portiere

print('portiere version:', portiere.__version__)

## 1 · The published results

`src/portiere/benchmarks/athena_icd_snomed/expected_results.json` is the source of truth. It is committed in the repo and used by the regression test `tests/test_benchmark_runner.py` to lock the numbers in CI.

In [ ]:
results_path = (
    Path(portiere.__file__).parent
    / 'benchmarks' / 'athena_icd_snomed' / 'expected_results.json'
)
results = json.loads(results_path.read_text())
print('Athena release:', results['athena_release_date'])
for run in results['runs']:
    print(
        f"  {run['backend']:<8} "
        f"top-1={run['top_1']:.3f}  "
        f"top-5={run['top_5']:.3f}  "
        f"top-10={run['top_10']:.3f}  "
        f"MRR={run['mrr']:.3f}  "
        f"(n={run['n']})"
    )

## 2 · Honest finding — BM25 wins

On ICD-10-CM → SNOMED, the gold mapping shares vocabulary with the source. An ICD description and its target SNOMED description have strong lexical overlap, so a tuned sparse retriever (BM25) beats both dense (SapBERT + FAISS) and the hybrid RRF combiner.

We publish all three rows so you can pick the right backend for **your** data. Dense and hybrid retrieval pay off on noisier free-text inputs (clinical notes, patient-described symptoms) where lexical overlap is low — those are not measured in v0.3.0.

## 3 · How to reproduce each row

On your own Athena export (free with registration at https://athena.ohdsi.org/), each row is a single CLI invocation:

```bash
# BM25 baseline (no embeddings, no GPU; ~minutes to index)
portiere benchmark athena-icd-snomed \
  --backend bm25s \
  --athena-dir /path/to/athena \
  --out runs/bm25.json

# Dense — downloads SapBERT (~440 MB on first run), builds a FAISS index
portiere benchmark athena-icd-snomed \
  --backend faiss \
  --athena-dir /path/to/athena \
  --out runs/faiss.json

# Hybrid — both indexes, RRF combiner (k=60)
portiere benchmark athena-icd-snomed \
  --backend hybrid \
  --athena-dir /path/to/athena \
  --out runs/hybrid.json
```

Each row appends to `expected_results.json` via `append_run_to_expected_results()`, upserting by backend key — re-running `--backend bm25s` overwrites only the BM25 row.

**Stratified sampling** (opt-in): add `--stratify-by domain` to sample proportionally across Athena domains instead of uniformly. Default uniform sampling preserves v0.2.1 comparability.

## 4 · Why not run the benchmark here?

The bundled vocab is too small to draw conclusions from:

In [ ]:
import csv

demo_concept = (
    Path(portiere.__file__).parent
    / '_demo_data' / 'vocabulary' / 'CONCEPT.csv'
)
with open(demo_concept, newline='') as f:
    rows = list(csv.DictReader(f, delimiter='\t'))

vocab_counts = {}
for r in rows:
    vocab_counts[r['vocabulary_id']] = vocab_counts.get(r['vocabulary_id'], 0) + 1

print('Bundled vocab row counts by vocabulary_id:')
for v, c in sorted(vocab_counts.items()):
    print(f'  {v}: {c}')
print()
print('Full Athena ICD10CM is ~94k concepts; SNOMED is ~370k.')

## 5 · Where the methodology lives

- **Runner & sampling:** `src/portiere/benchmarks/athena_icd_snomed/`
- **Schema of the results JSON:** `src/portiere/benchmarks/athena_icd_snomed/expected_results.json` (top-level `runs: []`)
- **Test set (gold concept_ids only — license-clean):** `src/portiere/benchmarks/athena_icd_snomed/gold_test_set.csv`
- **Regression test:** `tests/test_benchmark_runner.py::TestRunBenchmark`
- **Full doc:** [`docs/benchmarks/athena-icd-snomed.md`](../benchmarks/athena-icd-snomed.md)

The v0.3.x roadmap adds a **USAGI baseline** comparison row and additional vocabulary pairs (RxNorm → ATC, LOINC cross-vocab).